# Lecture 21 · DDPM on a 2D toy

Implement the full diffusion training loop on a 2D Swiss-roll · visualize the forward noise and the reverse denoising.

This is the cheapest way to understand diffusion · no U-Net, no CFG, just the core denoising-score-matching idea.

---

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

## 1 · The data · 2D Swiss roll

In [ ]:
def swiss_roll(n=2000, noise=0.02):
    t = torch.linspace(0, 4 * np.pi, n)
    r = 0.1 + t / (4 * np.pi) * 2.0
    x = r * torch.cos(t) + noise * torch.randn(n)
    y = r * torch.sin(t) + noise * torch.randn(n)
    return torch.stack([x, y], dim=-1).float()

data = swiss_roll().to(device)
plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0].cpu(), data[:, 1].cpu(), s=4, c='#7b9e89')
plt.title('target distribution · 2D Swiss roll')
plt.axis('equal')
plt.show()

## 2 · Noise schedule

Cosine schedule · smoother than linear near the endpoints.

In [ ]:
T = 200
s = 0.008

def cosine_alpha_bar(t, T, s=0.008):
    f = lambda t: np.cos(((t / T + s) / (1 + s)) * np.pi / 2) ** 2
    return f(t) / f(0)

ts = torch.arange(T + 1)
alpha_bar = torch.tensor([cosine_alpha_bar(t.item(), T) for t in ts], dtype=torch.float32).to(device)
alpha = alpha_bar[1:] / alpha_bar[:-1]
beta = 1 - alpha

plt.plot(alpha_bar.cpu(), label='α̅_t')
plt.plot(beta.cpu(), label='β_t')
plt.legend(); plt.title('cosine noise schedule'); plt.xlabel('t')
plt.show()

## 3 · Forward process · visualize

In [ ]:
def forward_sample(x0, t):
    # Closed form: x_t = sqrt(α̅_t) x_0 + sqrt(1-α̅_t) ε
    ab = alpha_bar[t]
    eps = torch.randn_like(x0)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps
    return x_t, eps

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t in zip(axes, [0, 50, 100, 150, 200]):
    x_t, _ = forward_sample(data, t)
    ax.scatter(x_t[:, 0].cpu(), x_t[:, 1].cpu(), s=2, c='#d97757')
    ax.set_title(f't = {t}   α̅={alpha_bar[t].item():.2f}')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.axis('equal')
plt.suptitle('Forward process · Swiss roll dissolves into N(0, I)')
plt.show()

## 4 · Tiny denoising network

MLP that takes `(x_t, t)` and predicts the noise ε.

In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, d_hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, 2),
        )

    def forward(self, x, t):
        # Normalize t into [0, 1]
        t_emb = (t.float() / T).unsqueeze(-1)
        inp = torch.cat([x, t_emb], dim=-1)
        return self.net(inp)

model = NoisePredictor().to(device)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

## 5 · Training loop

The DDPM loss · sample a random `t`, compute `x_t` in closed form, predict the noise, MSE against the true ε.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 256
steps = 3000

losses = []
for step in range(steps):
    # Sample a minibatch from data
    idx = torch.randint(0, len(data), (batch_size,))
    x0 = data[idx]

    # Sample a random t for each example
    t = torch.randint(1, T + 1, (batch_size,), device=device)

    # Forward process
    ab_t = alpha_bar[t].unsqueeze(-1)
    eps = torch.randn_like(x0)
    x_t = torch.sqrt(ab_t) * x0 + torch.sqrt(1 - ab_t) * eps

    # Predict the noise
    pred_eps = model(x_t, t)

    # MSE loss
    loss = ((pred_eps - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

    if step % 500 == 0:
        print(f'step {step:4d}  loss = {loss.item():.4f}')

plt.plot(losses); plt.yscale('log'); plt.title('DDPM training loss'); plt.show()

## 6 · Sample · reverse process from noise to data

In [ ]:
@torch.no_grad()
def sample(model, n, T=200, snapshot=False):
    x = torch.randn(n, 2, device=device)
    snapshots = []
    for t in range(T, 0, -1):
        t_batch = torch.full((n,), t, device=device)
        ab_t = alpha_bar[t]
        ab_prev = alpha_bar[t-1]
        beta_t = 1 - alpha[t-1]

        pred_eps = model(x, t_batch)
        # Posterior mean (simplified DDPM)
        mean = (x - (1 - alpha[t-1]) / torch.sqrt(1 - ab_t) * pred_eps) / torch.sqrt(alpha[t-1])

        if t > 1:
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mean

        if snapshot and t % 40 == 0:
            snapshots.append((t, x.clone()))
    return x, snapshots

samples, snaps = sample(model, 2000, snapshot=True)

fig, axes = plt.subplots(1, len(snaps) + 1, figsize=(18, 3))
for ax, (t, s) in zip(axes, snaps):
    ax.scatter(s[:, 0].cpu(), s[:, 1].cpu(), s=2, c='#d97757')
    ax.set_title(f't = {t}')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.axis('equal')
axes[-1].scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=2, c='#7b9e89')
axes[-1].set_title('final · t = 0')
axes[-1].set_xlim(-4, 4); axes[-1].set_ylim(-4, 4); axes[-1].axis('equal')
plt.suptitle('Reverse process · noise → Swiss roll')
plt.show()

## 7 · Comparison · real vs generated

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(data[:, 0].cpu(),    data[:, 1].cpu(),    s=3, c='#7b9e89')
axes[0].set_title('real'); axes[0].axis('equal')
axes[1].scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=3, c='#d97757')
axes[1].set_title('generated by DDPM'); axes[1].axis('equal')
plt.show()

**Reflection**

- Training loss = MSE between predicted and true noise. Simple.
- Forward is closed-form · no iteration needed during training.
- Reverse takes T = 200 steps at inference.
- The model generalizes: it never saw the exact swiss-roll curve labels, only pairs of `(x_t, ε)`.

**Next** · scale this up to images with a U-Net · add CFG · and you have Stable Diffusion.